# Pinecone

**Pinecone** is a **fully managed, serverless vector database** in the cloud. Unlike Chroma or FAISS, it is a closed-source commercial service built only for vector search. It is one of the most widely used vector databases for production RAG because:

- **Nothing to run yourself:** you create an index in the Pinecone console (or with a few lines of code) and connect to it with an API key. Pinecone handles the servers, scaling, backups and replication.
- **Data is saved automatically:** every chunk you add is stored in the cloud straight away. There is no `save_local()` or `dump()`, and the data survives kernel restarts.
- **Shared by many users and apps:** any notebook, API or app with an API key for the project sees the same data.
- **Namespaces:** one index can be split into separate partitions, for example one per user or per customer, and each search looks inside only one of them.
- **Filtering inside the database:** metadata filters run as part of the search, like Chroma's `where` and Astra DB's `filter`.
- **Extras for RAG:** Pinecone hosts its own embedding and reranking models, and supports **sparse vectors** for keyword and hybrid search.
- **LangChain support:** the `langchain-pinecone` package provides `PineconeVectorStore`, which works like every other LangChain vector store.

Like Astra DB in the previous notebook, Pinecone runs **in the cloud**, and every add or search is a request over the network. Two practical differences: Pinecone authenticates with a single **API key**, and the **index must exist before** LangChain can use it.

> **In one sentence:** Pinecone is a serverless cloud vector database you connect to with an API key, so your vectors are stored, searched and shared without running any servers yourself.

This notebook uses `langchain-pinecone` 0.2.x (`PineconeVectorStore`), which talks to Pinecone through the official `pinecone` Python SDK 7.x.

## Pinecone vs. the vector stores we've used so far

| | Pinecone | Astra DB | Chroma | FAISS | InMemoryVectorStore |
|---|---|---|---|---|---|
| **Where it runs** | In the cloud (AWS, Google Cloud or Azure), managed for you | In the cloud, managed for you | Your process, or a Chroma server | Your process | Your process |
| **Open source** | No: a cloud service only | No: a cloud service (built on open-source Cassandra) | Yes | Yes | Yes |
| **Setup** | Create an index, then `langchain-pinecone` + API key | Create a database online, then `langchain-astradb` + endpoint + token | `chromadb` + `langchain-chroma` | `faiss-cpu` | Nothing: part of `langchain-core` |
| **Who creates the index / collection** | **You**, before connecting | `AstraDBVectorStore`, the first time | `Chroma`, the first time | `FAISS.from_documents` | Nothing to create |
| **Saving** | Automatic, in the cloud | Automatic, in the cloud | Automatic with `persist_directory` | Manual: `save_local()` | Manual: `dump()` |
| **Search** | Approximate (Pinecone's serverless index) | Approximate (JVector graph index) | Approximate (HNSW) | Exact (Flat) or approximate | Exact |
| **Score from `similarity_search_with_score`** | Cosine similarity from -1 to 1: **higher = better** | Similarity from 0 to 1: higher = better | Distance: lower = better | Distance: lower = better | Cosine similarity: higher = better |
| **Metadata filtering** | Dict filters, **inside the database** | Dict filters, inside the database | Dict filters (`where`), inside the database | After the search (`fetch_k`) | Python function, before the search |
| **Adding an existing id** | Overwrites it (upsert) | Overwrites it (upsert) | Overwrites it (upsert) | Raises `ValueError` | Overwrites it (upsert) |
| **Several users / apps** | Yes: anyone with an API key for the project | Yes: anyone with the endpoint and a token | Yes, with a Chroma server | No | No |
| **Cost** | Free Starter plan, then pay as you go | Free tier, then pay as you go | Free (open source), or Chroma Cloud | Free (open source) | Free |
| **Best for** | Production apps that want a managed vector database with very little setup | Production apps that need a managed, shared, always-on vector database | Local apps that need CRUD and filtering | Fast local search over large collections | Learning, tests, small prototypes |

The retrievers and LCEL chains from the previous notebooks work unchanged: only the line that creates the vector store changes.

## The building blocks: project, index, namespace, record

```
Pinecone organization (your account)
 │
 └── Project "Default"                    API keys belong to a project
      │
      └── Index "rag-bootcamp"            serverless, in one cloud region
           │   dimension = 1536, metric = cosine        ← fixed when the index is created
           │   host: https://rag-bootcamp-<id>.svc.<region-id>.pinecone.io
           │
           ├── Namespace (default)        used when you don't pass `namespace`
           │     ├── record {"id": "chunk_0", "values": [...], "metadata": {"text": "...", "source": "..."}}
           │     ├── record {"id": "chunk_1", ...}
           │     └── ...
           │
           └── Namespace "user_42"        another partition of the same index
```

`PineconeVectorStore` stores each chunk as one record:

| Field | What it holds | Example |
|---|---|---|
| `id` | The chunk id | `"chunk_0"` |
| `values` | The embedding | 1536 numbers for `text-embedding-3-small` |
| `metadata` | The chunk's metadata **and its text**, under the key `text` | `{"text": "AI Agents: An Introduction ...", "source": "/.../RAG-Bootcamp/data/raw/text/ai_agents.txt", "topic": "agents"}` |
| `sparse_values` | An optional sparse vector, for hybrid search | `{"indices": [10, 45], "values": [0.5, 0.3]}` |

Pinecone has no separate field for the text. LangChain puts `page_content` into the metadata under `text` when it adds a chunk, and takes it out again when it returns search results. So the text counts towards the metadata size limit, and `text` should not be one of your own metadata keys (you can rename it with `text_key=`).

- **Project:** groups indexes and holds the API keys. The Starter plan has one project.
- **Index:** like an Astra DB collection. Every record has the **same dimension** and is searched with the **same metric**, both fixed when the index is created, along with the cloud and region. Each index has its own **host** URL, which the SDK looks up from the index name. The Starter plan allows 5 indexes, all in AWS `us-east-1`.
- **Namespace:** a partition inside an index. A search looks in **one namespace only**, so namespaces are an easy way to keep each user's or customer's data apart. A namespace is created automatically the first time you add records to it. The Starter plan allows 100 per index.
- **Record:** one chunk. The `metadata` dict is what you filter on.

## Connecting: API key and index

You need an API key and an index, both from the Pinecone console at [app.pinecone.io](https://app.pinecone.io):

1. **Sign up:** the free **Starter** plan is enough for this course.
2. **API key:** open *API keys* in your project and create a key. It starts with `pcsk_`.
3. **Create an index:** either in the console (*Create index*, with dimension `1536`, metric `cosine`, serverless on AWS `us-east-1`), or in code with the `pinecone` SDK:

```python
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone()   # reads PINECONE_API_KEY from the environment

if not pc.has_index("rag-bootcamp"):
    pc.create_index(
        name="rag-bootcamp",
        dimension=1536,                  # must match the embedding model: text-embedding-3-small
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
```

Put the key in your `.env` file, next to `OPENAI_API_KEY`:

```
PINECONE_API_KEY=pcsk_...
PINECONE_INDEX_NAME=rag-bootcamp      # optional
```

`PineconeVectorStore` reads these two environment variables when you don't pass `pinecone_api_key` or `index_name`. Install the package with `uv add langchain-pinecone`. It also installs the `pinecone` SDK.

> **Create the index first.** Unlike `AstraDBVectorStore` and `Chroma`, `PineconeVectorStore` **does not create the index**. Create it before connecting, with a dimension that matches your embedding model.

> **Python 3.14:** `langchain-pinecone` 0.2.13 officially supports Python up to 3.13 and pins the `pinecone` SDK to 7.x. `uv add` still installs it in our Python 3.14 environment, so use `uv` rather than `pip` here.

> **The API key is a password.** Anyone who has it can read and delete every index in the project. Keep it in `.env` (which git ignores) and never paste it into a notebook cell.

## Three ways to embed: OpenAI, Pinecone's hosted models, or integrated embedding

Pinecone hosts its own embedding models (**Pinecone Inference**), such as `multilingual-e5-large` and `llama-text-embed-v2` (both 1024 dimensions), so you can also embed without an OpenAI key.

```
Client-side: OpenAIEmbeddings or PineconeEmbeddings      Integrated embedding: pc.create_index_for_model(...)

chunk text                                                chunk text
   │  embedded by a call from your notebook                  │  sent as plain text
   ▼                                                         ▼
vector ──► Pinecone stores it in "values"                 the index calls its embedding model itself
                                                          and stores the result
```

| | `OpenAIEmbeddings` | `PineconeEmbeddings` | Integrated embedding |
|---|---|---|---|
| **Who calls the embedding model** | Your code calls OpenAI | Your code calls Pinecone Inference | The Pinecone index |
| **Set up with** | `embedding=OpenAIEmbeddings(...)` | `embedding=PineconeEmbeddings(model="multilingual-e5-large")` | `pc.create_index_for_model(..., embed={...})` |
| **API keys** | `OPENAI_API_KEY` + `PINECONE_API_KEY` | `PINECONE_API_KEY` only | `PINECONE_API_KEY` only |
| **Index dimension** | 1536 (`text-embedding-3-small`) | 1024 | Set by the model |
| **Works with `PineconeVectorStore`** | Yes | Yes | No: use the `pinecone` SDK (`upsert_records`, `search`) |

```python
# Integrated embedding: the index embeds the "chunk_text" field itself
pc.create_index_for_model(
    name="rag-bootcamp-integrated",
    cloud="aws",
    region="us-east-1",
    embed={"model": "llama-text-embed-v2", "field_map": {"text": "chunk_text"}},
)
index = pc.Index("rag-bootcamp-integrated")
index.upsert_records("docs", [{"_id": "chunk_0", "chunk_text": "AI Agents: An Introduction ...", "topic": "agents"}])
results = index.search(namespace="docs", query={"inputs": {"text": "What is an AI agent?"}, "top_k": 3})
```

This notebook uses **client-side** `OpenAIEmbeddings`, with the same `text-embedding-3-small` model as the previous notebooks, so the index has **1536** dimensions.

## Similarity metrics and scores

The **metric** is chosen when the index is created and can't be changed afterwards:

| `metric` | Notes |
|---|---|
| `"cosine"` | **The default.** Works for any embeddings |
| `"dotproduct"` | Same ranking as cosine for **normalised** vectors (OpenAI embeddings are normalised). **Required for hybrid search** with dense + sparse vectors in one index |
| `"euclidean"` | Based on the straight-line distance between the vectors |

Pinecone serverless indexes use Pinecone's own **approximate** nearest-neighbour index, with storage and compute managed separately, so searches stay fast with millions or billions of vectors.

**Scores:** `similarity_search_with_score` returns Pinecone's **raw score**. With `cosine`, that is the **cosine similarity from -1 to 1: higher = better**, the same numbers `InMemoryVectorStore` gave us.

**Relevance scores:** `similarity_search_with_relevance_scores` and `as_retriever(search_type="similarity_score_threshold")` rescale it to 0–1, with the same formula Astra DB uses:

$$\text{relevance score} = \frac{1 + \text{cosine similarity}}{2}$$

| Cosine similarity (`..._with_score`) | Relevance score (`..._with_relevance_scores`) | Meaning |
|---|---|---|
| `1.0` | `1.0` | Same meaning |
| `0.68` | `0.84` | A good match, like the top chunk for *"What is reinforcement learning?"* in the previous notebooks |
| `0` | **`0.5`** | Unrelated |
| `-1.0` | `0` | Opposite |

So the two methods give **different numbers for the same chunk**. A relevance threshold of `0.8` means "cosine similarity above 0.6". Because 0.5 already means "unrelated", choose a threshold well above 0.5, for example:

```python
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 3, "score_threshold": 0.8},
)
```

> `PineconeVectorStore` doesn't read the metric from the index: it assumes `cosine` when it computes relevance scores. If you create a `dotproduct` or `euclidean` index, pass a matching `distance_strategy`. The raw scores from `similarity_search_with_score` are not affected.

## Working with PineconeVectorStore in LangChain

| Method | What it does |
|---|---|
| `PineconeVectorStore(index_name=..., embedding=..., namespace=None)` | Connect to an **existing** index |
| `PineconeVectorStore.from_documents(docs, embedding, ids=..., index_name=..., namespace=...)` | Connect and add the chunks in one step |
| `vectorstore.add_documents(docs, ids=..., namespace=...)` | Embed and add chunks. An id that already exists is **overwritten** |
| `vectorstore.similarity_search(query, k=4, filter=..., namespace=...)` | Return the `k` most similar `Document`s (default `k=4`) |
| `vectorstore.similarity_search_with_score(query, k=4, filter=..., namespace=...)` | Return `(Document, score)` pairs, with cosine similarities from -1 to 1 |
| `vectorstore.similarity_search_with_relevance_scores(query, k=4, ...)` | Return `(Document, score)` pairs, with scores rescaled to 0–1 |
| `vectorstore.max_marginal_relevance_search(query, k, fetch_k, filter=..., namespace=...)` | Return relevant **and** varied chunks (MMR) |
| `vectorstore.delete(ids=[...])` | Remove chunks by id |
| `vectorstore.delete(filter={...})` | Remove every chunk that matches a filter |
| `vectorstore.delete(delete_all=True, namespace=...)` | Remove all chunks in a namespace but keep the index |
| `vectorstore.as_retriever(search_kwargs={"k": 3})` | Use it in a RAG chain, like the other retrievers |

Every method also has an async version: `aadd_documents`, `asimilarity_search`, `adelete`, ...

`PineconeVectorStore` doesn't wrap everything: there is no `get_by_ids` and no metadata update. For those, use the SDK's index object, `vectorstore.index`, and the client `pc = Pinecone()`:

| SDK call | What it does |
|---|---|
| `vectorstore.index.describe_index_stats()` | The dimension and the number of records in each namespace |
| `vectorstore.index.fetch(ids=[...])` | Fetch records by id (the text is in `metadata["text"]`) |
| `vectorstore.index.update(id=..., set_metadata={...})` | Add or change metadata keys of one record |
| `vectorstore.index.list(prefix="chunk_")` | List record ids, page by page |
| `pc.list_indexes()`, `pc.describe_index(name)` | See your indexes and their settings |
| `pc.delete_index(name)` | Delete a whole index and its data |

### Connect, add and reconnect

```python
import os
from langchain_pinecone import PineconeVectorStore

# Connect to the index (create it first, see above)
vectorstore = PineconeVectorStore(
    index_name="rag-bootcamp",
    embedding=embeddings,
    pinecone_api_key=os.environ["PINECONE_API_KEY"],
)

# Add the chunks: they are stored in the cloud straight away
vectorstore.add_documents(chunks, ids=[f"chunk_{i}" for i in range(len(chunks))])

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
```

Later, in another notebook or app, run the same `PineconeVectorStore(...)` call: it connects to the existing index, and the chunks are already there. There is nothing to save or load.

### Things to watch out for

- **Create the index first, with the right dimension.** `PineconeVectorStore` doesn't create it, and adding 1536-number vectors to a 1024-dimension index fails with a dimension-mismatch error.
- **Settings are fixed per index.** The dimension, metric, cloud and region are set when the index is created. Changing any of them means creating a new index and adding the data again.
- **Fixed ids prevent duplicates.** `add_documents` with ids that already exist replaces those chunks. Without ids, every run creates new random UUIDs, so running the cell twice **duplicates** your chunks, and they stay in the cloud.
- **New records take a moment to show up.** Pinecone is **eventually consistent**: right after `add_documents`, a search or `describe_index_stats()` may not see the new chunks yet. Wait a few seconds before searching.
- **Strict metadata rules.** Values must be strings, numbers, booleans or lists of strings. `None` values and nested dicts are rejected, so drop those keys before adding. Numbers come back as floats (`page: 3` becomes `3.0`), and each record's metadata, chunk text included, must stay under **40 KB**.
- **One namespace per search.** Chunks added with `namespace="docs"` are invisible to a search in the default namespace. Pass the same `namespace` when adding and searching, or set it once with `PineconeVectorStore(namespace=...)`.
- **Every call goes over the network.** Each search embeds the question and then sends a request to Pinecone, so it is slower than an in-process FAISS search. `add_documents` sends the chunks in batches (`batch_size=32` by default).
- **Same embedding model.** As with every vector store, query with the model that built the index.
- **Deleting is permanent.** `delete(delete_all=True)` and `pc.delete_index()` remove the data in the cloud, for every user of the index. You can protect an index with `deletion_protection="enabled"`.

## Metadata filtering

Filters are **dicts** with MongoDB-style operators, and they run **inside the database** as part of the search. You get `k` results whenever at least `k` chunks match, and there is no `fetch_k` to tune:

```python
# One topic
results = vectorstore.similarity_search("What is reinforcement learning?", k=3, filter={"topic": "ml"})

# ml or dl
results = vectorstore.similarity_search(
    "What is reinforcement learning?",
    k=3,
    filter={"topic": {"$in": ["ml", "dl"]}},
)

# The same filter in a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3, "filter": {"topic": "dl"}})
```

| Operator | Meaning | Example |
|---|---|---|
| `$eq`, `$ne` | equal / not equal | `{"topic": "ml"}` (short for `$eq`) |
| `$gt`, `$gte`, `$lt`, `$lte` | greater / less than (numbers only) | `{"page": {"$gte": 3}}` |
| `$in`, `$nin` | in / not in a list | `{"topic": {"$in": ["ml", "dl"]}}` |
| `$exists` | the field is present | `{"topic": {"$exists": True}}` |
| `$and`, `$or` | combine conditions | `{"$or": [{"topic": "ml"}, {"topic": "dl"}]}` |

For a metadata field that holds a **list of strings**, such as `"tags": ["rag", "faiss"]`, the filter `{"tags": "rag"}` matches the records whose list **contains** `"rag"`.

Keys are sent as they are: Pinecone keeps metadata keys at the top level of the record, so unlike Astra DB there is no `metadata.` prefix. The same filters also work with `delete(filter=...)`.

How this compares with the other stores:

| | Where the filter runs | Can you get fewer than `k` results? |
|---|---|---|
| FAISS | In Python, **after** the search, on the `fetch_k` nearest chunks | Yes, when few of those chunks match |
| InMemoryVectorStore | In Python, **before** the search | Only when fewer than `k` chunks match |
| Astra DB | **Inside the database**, during the search | Only when fewer than `k` chunks match |
| Pinecone | **Inside the database**, during the search | Only when fewer than `k` chunks in the namespace match |

## Hybrid search

Vector search finds chunks with a **similar meaning**, but it can miss exact words such as product codes, error messages or names. Pinecone handles keywords with **sparse vectors**, which give a weight to each important word instead of storing 1536 numbers for the meaning:

```
question ─┬─► dense vector search    (meaning)   ─┐
          │                                      ├─► combine ─► rerank (optional) ─► top k
          └─► sparse vector search   (keywords)  ─┘
```

There are two ways to set it up:

- **One index, dense + sparse:** a `dotproduct` index can store a dense and a sparse vector in each record and score both in one query.
- **Two indexes:** a dense index plus a **sparse index** (`vector_type="sparse"`), searched separately and then merged. `langchain-pinecone` provides `PineconeSparseEmbeddings` (model `pinecone-sparse-english-v0`), `PineconeSparseVectorStore` and `PineconeRerank` (default model `bge-reranker-v2-m3`) for this.

We'll look at hybrid search in detail later in the course.

## Strengths and limits

**Strengths**

- Fully managed and serverless: no servers, scaling or backups to handle
- Very little setup: one API key and one index, and you're ready
- Data is saved automatically and shared by every app and user with access
- Scales to billions of vectors, with storage and compute managed separately
- Namespaces keep each user's or customer's data apart inside one index
- Metadata filters run inside the database, during the search
- Upsert, delete by id, by filter or by namespace, and metadata updates through the SDK
- Hosted embedding and reranking models, and sparse vectors for keyword and hybrid search
- Runs on AWS, Google Cloud and Azure, with a free Starter plan to start

**Limits to keep in mind**

- **Needs an account and the internet.** Every call is a network request, and your app depends on a cloud service.
- **Closed source, cloud only.** You can't host it yourself (Pinecone Local is only an emulator for tests), so your app is tied to Pinecone's service.
- **Slower per query than in-process stores.** A network round trip costs more than a FAISS or in-memory search, which matters for small local projects.
- **Costs grow with use.** Beyond the Starter plan, you pay for reads, writes and storage. The Starter plan is limited to 5 indexes, 100 namespaces per index, 2 GB of storage, monthly read and write quotas, and AWS `us-east-1`.
- **Eventual consistency.** New or changed records can take a few seconds to show up in searches.
- **Strict metadata.** No `None` values or nested dicts, and at most 40 KB per record, chunk text included.
- **Fixed index settings.** Changing the dimension, metric or embedding model means creating a new index and adding the data again.
- **The LangChain wrapper lags behind the SDK.** `langchain-pinecone` 0.2.x pins the `pinecone` SDK to 7.x, and creating indexes or fetching and updating records still needs the SDK.
- **Secrets to manage.** The API key gives access to every index in the project, so it must stay out of your code and out of git.

## Key takeaways

- **Pinecone** is a closed-source, serverless **cloud** vector database. `PineconeVectorStore` from `langchain-pinecone` plugs it into LangChain.
- You connect with an **API key**, kept in `.env` as `PINECONE_API_KEY`, and an **index name** (`index_name=` or `PINECONE_INDEX_NAME`).
- **Create the index first:** `PineconeVectorStore` doesn't create it. Its **dimension** and **metric** are fixed and must suit the embedding model (`1536` and `cosine` for `text-embedding-3-small`).
- Each record holds an `id`, the vector (`values`) and `metadata`. LangChain stores the chunk text in the metadata under `text`.
- **Namespaces** split one index into partitions, and every search looks inside one namespace.
- **Saving is automatic** and the data is **shared**, but Pinecone is **eventually consistent**: new chunks can take a few seconds to appear in searches.
- `similarity_search_with_score` returns the **cosine similarity (-1 to 1, higher = better)**. Relevance scores rescale it to 0–1, so unrelated chunks score about **0.5**.
- `add_documents` with existing ids **overwrites** them, so fixed ids prevent duplicates.
- **Metadata filters** are dicts with MongoDB-style operators, run **inside the database**. Metadata values can't be `None` or nested dicts.
- Pinecone also hosts **embedding** and **reranking** models, and supports **sparse vectors** for hybrid search.
- `as_retriever()` plugs Pinecone into the same LCEL and conversational RAG chains we built with Chroma, FAISS, `InMemoryVectorStore` and Astra DB.

**Next:** we'll create a Pinecone index, add our chunks to it, search them with and without filters, and use the index in a RAG chain.

In [1]:
## Setup: shared project paths from ragkit.config, which also loads the API keys from .env
import os

from ragkit.config import PROJECT_ROOT, RAW_DATA_DIR

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp


In [2]:
## Load libraries
import warnings
warnings.filterwarnings("ignore")

import json
import time   # short pauses while Pinecone makes new records searchable
from collections import defaultdict

import pandas as pd
from pinecone import Pinecone, ServerlessSpec

# LangChain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough, RunnableBranch
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# LangChain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_pinecone import PineconeVectorStore
from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    Docx2txtLoader,
    CSVLoader,
    JSONLoader,
    SQLDatabaseLoader,
)
from langchain_community.utilities import SQLDatabase

In [3]:
from pinecone import Pinecone

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))



In [4]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1024)

print(len(embeddings.embed_query("test")))   # should print 1024, matching the index


1024


In [5]:
INDEX_NAME="dev-index-2"

if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=1024,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(INDEX_NAME)

print(index.describe_index_stats())

{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 44}},
 'total_vector_count': 44,
 'vector_type': 'dense'}


In [6]:
vectorstore = PineconeVectorStore(index=index, embedding=embeddings)

In [7]:
## Load the .txt files, split them, and add the chunks to Pinecone
docs = []
for path in sorted(RAW_DATA_DIR.rglob("*.txt")):
    docs.extend(TextLoader(str(path), encoding="utf-8").load())

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

ids = [f"chunk_{i}" for i in range(len(chunks))]   # fixed ids: re-running overwrites instead of duplicating
vectorstore.add_documents(chunks, ids=ids)

time.sleep(5)   # new records take a few seconds to become searchable
print(index.describe_index_stats())


{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 44}},
 'total_vector_count': 44,
 'vector_type': 'dense'}


In [8]:
## Retrieval: cosine similarity, higher = better
results = vectorstore.similarity_search_with_score("What is an AI agent?", k=3)
for doc, score in results:
    print(f"{score:.3f} | {doc.metadata['source']} | {doc.page_content[:100]}...")


0.789 | /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text/ai_agents.txt | AI Agents: An Introduction

An AI agent is a system that uses a large language model (LLM) to decide...
0.671 | /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text/ai_agents.txt | Applications
AI agents are used for coding assistants that read, write, and test code, customer supp...
0.597 | /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text/ai_agents.txt | Conclusion
AI agents turn LLMs from question-answering systems into systems that can take actions. B...


In [9]:
## RAG: retriever + prompt + LLM
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
llm = ChatOpenAI(model="gpt-5", temperature=0)

prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question.
If you don't know the answer based on the context, say you don't know.

Context:
{context}

Question: {question}

Answer:""")

rag_chain = (
    {"context": retriever | (lambda docs: "\n\n".join(d.page_content for d in docs)),
     "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("What is an AI agent?"))


An AI agent is a system that uses a large language model to decide what to do to reach a goal, planning multi-step actions, calling tools (like web search or a calculator), checking results, and continuing until the task is finished—the LLM is the “brain” and tools are the “hands.”


In [10]:
## RAG with a similarity score threshold: only chunks with a relevance score >= 0.7
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 4, "score_threshold": 0.4},   # k is now a maximum
)

# See how many chunks pass the threshold
for question in ["What is an AI agent?", "What is the capital of France?","What is Attention mechanism"]:
    print(f"{question} -> {len(retriever.invoke(question))} chunks")

llm = ChatOpenAI(model="gpt-5", temperature=0)

prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question.
If you don't know the answer based on the context, say you don't know.

Context:
{context}

Question: {question}

Answer:""")

rag_chain = (
    {"context": retriever | (lambda docs: "\n\n".join(d.page_content for d in docs)),
     "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("What is an AI agent?"))
print(rag_chain.invoke("What is the capital of France?"))


What is an AI agent? -> 4 chunks
What is the capital of France? -> 4 chunks
What is Attention mechanism -> 4 chunks
An AI agent is a system that uses a large language model to decide what to do to reach a goal—planning multi-step actions, calling tools (like web search or a calculator), checking results, and continuing until the task is finished, with the LLM as the “brain” and tools as the “hands.”
I don’t know based on the provided context.
